# 4.2 — Contribution of Temporal Architecture and Spatial Attention

Ablation chain: LSTM (per-station recurrence) → Spatially Blind
Transformer (temporal attention only) → Dense (full
factorised attention). All at MR=0.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

RUNS  = C.discovered_runs()
MR0_RUNS = [r for r in RUNS if "mr0.00" in RUNS[r]]
ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
AGG  = {r: C.load_agg(r, "mr0.00") for r in MR0_RUNS}
GRID = AGG[MR0_RUNS[0]]["grid"]; LEAD = C.lead_labels(GRID); K = len(GRID)
NV = len(VARS)
print("Models at MR=0.00:", MR0_RUNS)

# Ablation chain
# Free AGG entries not in the ablation chain after CHAIN is defined
_NEEDED_42 = set()
CHAIN = ["v32-blind", "v31"]
CHAIN_LABELS = [C.MODELS[r][0] for r in CHAIN]
CHAIN_COLORS = [C.MODELS[r][1] for r in CHAIN]
# Reference lead ≈ 3 h → index 6
REF_KI = 6; REF_LEAD = LEAD[REF_KI]
print(f"Reference lead: {REF_LEAD} (index {REF_KI})")
# Free unused AGG entries
_needed = set(CHAIN) | {'v32-blind', 'v31', 'v27'}
for _r in list(AGG):
    if _r not in _needed:
        del AGG[_r]
import gc; gc.collect()


## MAE vs lead — ablation chain

## Spatial-attention gain: MAE(Blind) − MAE(Dense)

Positive = Dense is better (spatial attention helps).
This is the most directly interpretable spatial-attention result.

In [ ]:
fig, axes = plt.subplots(1, NV, figsize=(17, 3.3))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    m_blind = C.metric(AGG["v32-blind"], "mae", pool=("N",))[:, vi]
    m_dense = C.metric(AGG["v31"], "mae", pool=("N",))[:, vi]
    gain = m_blind - m_dense
    ax.bar(range(1, K), gain[1:], color="#5FAF5F", edgecolor="k", lw=0.4, alpha=0.7)
    ax.axhline(0, ls=":", color="k", lw=0.8)
    ax.set_xticks(range(1, K, 2))
    ax.set_xticklabels(LEAD[1::2], rotation=45, ha="right", fontsize=6.5)
    ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10); ax.grid(alpha=.3)
axes[0].set_ylabel("MAE(Blind) − MAE(Dense)")
fig.suptitle("Spatial-attention gain: MAE(Spatially Blind) − MAE(Dense) vs lead time, MR=0, all stations", y=1.04)
plt.tight_layout(); C.save_fig(fig, "42_spatial_attn_gain"); plt.show()
plt.close(fig)

# Numerical summary at REF_KI
print(f"\nSpatial-attention gain at {REF_LEAD}:")
for vi, v in enumerate(VARS):
    mb = C.metric(AGG["v32-blind"], "mae", pool=("N",))[REF_KI, vi]
    md = C.metric(AGG["v31"], "mae", pool=("N",))[REF_KI, vi]
    print(f"  {v:>12s}: {mb - md:+.3f} {C.UNITS[v]}  "
          f"({100*(mb - md)/mb:+.1f}%)")

### Spatial-attention gain on the map

Same quantity as above (MAE(Blind) − MAE(Dense) at the reference lead),
per station, plotted spatially instead of pooled across the network.
Positive (green) = Dense wins at that station, i.e. spatial
attention helps there; negative (purple) = the Spatially Blind
does better locally.

In [ ]:
# ── DEM + Swiss border, for the map below (same as 47_topographic_controls) ──
import geopandas as gpd
import rioxarray  # noqa

PROJ = os.path.abspath(os.path.join(os.getcwd(), "..", "..")) \
       if os.path.isfile("common.py") else os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

PATH_SWISSSHAPE = os.path.expanduser(
    os.environ.get("SWISSSHAPE",
        os.path.join(PROJ, "swissboundaries3d_2056.shp.zip")))
_CAND = [os.environ.get("DATA_ROOT", ""),
         os.path.expanduser("~/PeakWeatherDataset"),
         os.path.join(PROJ, "PeakWeatherDataset")]
DATA_ROOT = next((p for p in _CAND if os.path.isdir(str(p))), _CAND[-1])

from peakweather.dataset import PeakWeatherDataset
ds_topo = PeakWeatherDataset(
    root=DATA_ROOT,
    parameters=["temperature", "pressure", "humidity",
                 "wind_speed", "wind_direction", "precipitation"],
    compute_uv=True, station_type="meteo_station",
    imputation_method=None, freq="d", extended_topo_vars="DEM")

def _load_dem_and_border(ds_topo, path_swissshape, coarsen=10):
    switzerland = gpd.read_file(
        path_swissshape,
        layer='swissBOUNDARIES3D_1_5_TLM_LANDESGEBIET').to_crs('EPSG:2056')
    minx, miny, maxx, maxy = switzerland.total_bounds
    topo = ds_topo.load_topography()
    dem  = topo['topo_DEM'].dem
    dem_ch = dem.rio.clip(switzerland.geometry, switzerland.crs, drop=False)
    dem_bg = dem.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_fg = dem_ch.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_bg = dem_bg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    dem_fg = dem_fg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    return dem_bg, dem_fg, switzerland

def draw_dem(ax, dem_bg, dem_fg, switzerland):
    norm = mcolors.Normalize(vmin=0, vmax=4500)
    dem_bg.plot(ax=ax, cmap='terrain', norm=norm, alpha=0.35,
                robust=True, add_labels=False, add_colorbar=False)
    dem_fg.plot(ax=ax, cmap='terrain', norm=norm,
                robust=True, add_labels=False, add_colorbar=False)
    switzerland.boundary.plot(ax=ax, color='white', linewidth=1.0)
    ax.axis('off')

print('Loading DEM + border ...')
dem_bg, dem_fg, switzerland = _load_dem_and_border(ds_topo, PATH_SWISSSHAPE)
print('Done.')

## Exclusion overlay

Every per-station scatter map below also marks stations excluded for that
variable (`common.excluded_station_variable_reasons()`) with an X:
**black** = missing in both train and test, **white** = missing in train
only, **grey** = missing in test only, **orange** = present in both splits
but excluded for a data-quality reason (BIZ/pressure, sensor drift).

In [ ]:
REASON_STYLE = {
    "missing_train": dict(color="white",   label="missing in train only"),
    "missing_test":  dict(color="0.6",     label="missing in test only"),
    "missing_both":  dict(color="black",   label="missing in train & test"),
    "drift":         dict(color="#E8A838", label="excluded \u2014 sensor drift"),
}
EXCL_REASONS = C.excluded_station_variable_reasons()

def overlay_exclusion_markers(ax, variable, s=70, legend=False):
    return  # X markers removed per user request

In [ ]:
# ── MAE of Dense (v31) per station at +3h by TOD, MR=0.0 ────────────────────
TOD_LABELS = ["00–06 UTC", "06–12 UTC", "12–18 UTC", "18–24 UTC"]
EXT = {r: dict(np.load(C.aggregate_extended(r, "mr0.00"))) for r in ["v31", "v32-blind", "v27"]}
e_dense = EXT["v31"]

fig, axes = plt.subplots(len(TOD_LABELS), NV,
                         figsize=(6.0 * NV, 3.5 * len(TOD_LABELS)), squeeze=False)
for ti, tod_label in enumerate(TOD_LABELS):
    for vi, v in enumerate(VARS):
        ax = axes[ti, vi]
        draw_dem(ax, dem_bg, dem_fg, switzerland)

        cnt_d = e_dense["tod_mod_cnt"][ti, REF_KI, :, vi]
        s_d   = e_dense["tod_mod_sum"][ti, REF_KI, :, vi]
        mae_d = np.where(cnt_d > 0, s_d / np.maximum(cnt_d, 1), np.nan)

        valid = ~np.isnan(mae_d)
        sc = ax.scatter(stn.easting[valid], stn.northing[valid], c=mae_d[valid],
                        s=45, cmap="magma", edgecolors="k", linewidths=0.3, zorder=5)
        fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02,
                     label=f"MAE [{C.UNITS[v]}]")
        if ti == 0:
            ax.set_title(f"{v} at {REF_LEAD}", fontsize=11)
        if vi == 0:
            ax.text(-0.06, 0.5, tod_label, transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontsize=10)

# ── Sync color scale per variable column ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(TOD_LABELS))]
    col_sc = [ax.collections[-1] for ax in col_axes if ax.collections]
    all_vals = np.concatenate([s.get_array() for s in col_sc])
    vmin, vmax = np.nanmin(all_vals), np.nanmax(all_vals)
    for s in col_sc:
        s.set_clim(vmin, vmax)

# ── Sync spatial axes per variable column ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(TOD_LABELS))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind component color scales ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
for ri in range(axes.shape[0]):
    sc_u = axes[ri, wu_i].collections[-1]
    sc_v = axes[ri, wv_i].collections[-1]
    vmin = min(sc_u.get_clim()[0], sc_v.get_clim()[0])
    vmax = max(sc_u.get_clim()[1], sc_v.get_clim()[1])
    sc_u.set_clim(vmin, vmax)
    sc_v.set_clim(vmin, vmax)

plt.tight_layout()
C.save_fig(fig, "42_dense_mae_map_tod")
plt.show()
plt.close(fig)


### Spatial-attention gain on the map, by time of day

Same map as above, faceted by UTC time-of-day bin (rows) instead of pooled
over the full test period. Colour scale is shared across the four bins
within each variable (column), so a station's colour can be compared
across rows directly — differences between rows are not a scale artefact.

In [ ]:
# ── Spatial-attention gain map by time-of-day bin ───────────────────────────
TOD_LABELS = ["00\u201306 UTC", "06\u201312 UTC", "12\u201318 UTC", "18\u201324 UTC"]
EXT_GAIN = {r: EXT[r] for r in ["v32-blind", "v31"]}  # reuse from EXT

# Per (tod bin, variable) gain, computed once so the colour scale can be
# shared across bins within a variable (pooling all 4 bins' finite values).
gain_by_bin = {}
for ti in range(len(TOD_LABELS)):
    row = []
    for vi in range(NV):
        cnt_b = EXT_GAIN["v32-blind"]["tod_mod_cnt"][ti, REF_KI, :, vi]
        s_b   = EXT_GAIN["v32-blind"]["tod_mod_sum"][ti, REF_KI, :, vi]
        mae_b = np.where(cnt_b > 0, s_b / np.maximum(cnt_b, 1), np.nan)

        cnt_d = EXT_GAIN["v31"]["tod_mod_cnt"][ti, REF_KI, :, vi]
        s_d   = EXT_GAIN["v31"]["tod_mod_sum"][ti, REF_KI, :, vi]
        mae_d = np.where(cnt_d > 0, s_d / np.maximum(cnt_d, 1), np.nan)

        row.append(mae_b - mae_d)                 # positive = Dense wins locally
    gain_by_bin[ti] = row

vmax_per_var = []
for vi in range(NV):
    allv = np.concatenate([gain_by_bin[ti][vi][~np.isnan(gain_by_bin[ti][vi])]
                           for ti in range(len(TOD_LABELS))])
    vmax_per_var.append(np.nanpercentile(np.abs(allv), 98) if allv.size else 1.0)

# ── Sync wind component color scales ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
wind_vmax = max(vmax_per_var[wu_i], vmax_per_var[wv_i])
vmax_per_var[wu_i] = vmax_per_var[wv_i] = wind_vmax

fig, axes = plt.subplots(len(TOD_LABELS), NV,
                         figsize=(6.0 * NV, 3.5 * len(TOD_LABELS)), squeeze=False)
for ti, tod_label in enumerate(TOD_LABELS):
    for vi, v in enumerate(VARS):
        ax = axes[ti, vi]
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        gain = gain_by_bin[ti][vi]
        valid = ~np.isnan(gain)
        vmax = vmax_per_var[vi]
        sc = ax.scatter(stn.easting[valid], stn.northing[valid], c=gain[valid],
                        s=45, cmap="PRGn", vmin=-vmax, vmax=vmax,
                        edgecolors="k", linewidths=0.3, zorder=5)
        overlay_exclusion_markers(ax, v, legend=(ti == 0 and vi == 0))
        fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label=f"[{C.UNITS[v]}]")
        if ti == 0:
            ax.set_title(f"{v} at {REF_LEAD}", fontsize=11)
        if vi == 0:
            ax.text(-0.06, 0.5, tod_label, transform=ax.transAxes, rotation=90,
                   va="center", ha="center", fontsize=10)

#fig.suptitle("Spatial-attention gain by station and time of day (green = "
#             "spatial attention helps, MR=0)", y=1.005)
plt.tight_layout()
# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(TOD_LABELS))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)
C.save_fig(fig, "42_spatial_attn_gain_map_tod"); plt.show()
plt.close(fig)

In [ ]:
# ── MAE gain: v27 vs Dense (v31) by time-of-day bin ─────────────────────────
# gain = MAE(v27) − MAE(v31): positive means Dense (v31) is better
if "v27" not in EXT:
    EXT["v27"] = dict(np.load(C.aggregate_extended("v27", "mr0.00")))

gain27_by_bin = {}
for ti in range(len(TOD_LABELS)):
    row = []
    for vi in range(NV):
        cnt_d = EXT["v31"]["tod_mod_cnt"][ti, REF_KI, :, vi]
        s_d   = EXT["v31"]["tod_mod_sum"][ti, REF_KI, :, vi]
        mae_d = np.where(cnt_d > 0, s_d / np.maximum(cnt_d, 1), np.nan)

        cnt_v = EXT["v27"]["tod_mod_cnt"][ti, REF_KI, :, vi]
        s_v   = EXT["v27"]["tod_mod_sum"][ti, REF_KI, :, vi]
        mae_v = np.where(cnt_v > 0, s_v / np.maximum(cnt_v, 1), np.nan)

        row.append(mae_v - mae_d)  # positive = Dense wins locally
    gain27_by_bin[ti] = row

vmax27 = []
for vi in range(NV):
    allv = np.concatenate([gain27_by_bin[ti][vi][~np.isnan(gain27_by_bin[ti][vi])]
                           for ti in range(len(TOD_LABELS))])
    vmax27.append(np.nanpercentile(np.abs(allv), 98) if allv.size else 1.0)

# ── Sync wind component color scales ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
wind_vmax = max(vmax27[wu_i], vmax27[wv_i])
vmax27[wu_i] = vmax27[wv_i] = wind_vmax

fig, axes = plt.subplots(len(TOD_LABELS), NV,
                         figsize=(6.0 * NV, 3.5 * len(TOD_LABELS)), squeeze=False)
for ti, tod_label in enumerate(TOD_LABELS):
    for vi, v in enumerate(VARS):
        ax = axes[ti, vi]
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        gain = gain27_by_bin[ti][vi]
        valid = ~np.isnan(gain)
        vmax = vmax27[vi]
        sc = ax.scatter(stn.easting[valid], stn.northing[valid], c=gain[valid],
                        s=45, cmap="PRGn", vmin=-vmax, vmax=vmax,
                        edgecolors="k", linewidths=0.3, zorder=5)
        fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02,
                     label=f"[{C.UNITS[v]}]")
        if ti == 0:
            ax.set_title(f"{v} at {REF_LEAD}", fontsize=11)
        if vi == 0:
            ax.text(-0.06, 0.5, tod_label, transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontsize=10)

# ── Sync axes per variable column ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(TOD_LABELS))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

fig.suptitle("MAE Transformer vs Dense MAE gain by station and TOD\n"
             "(green = Dense better, MR=0.0)", y=1.005)
plt.tight_layout()
C.save_fig(fig, "42_v27_vs_dense_gain_map_tod")
plt.show()
plt.close(fig)

In [ ]:
# ── Morning 06–12 UTC maps: Dense MAE, SA gain, masking gain ─────────────────
SEL_VARS = ["temperature", "pressure", "humidity", "wind_u"]
SEL_VI = [VARS.index(v) for v in SEL_VARS]
N_SV = len(SEL_VARS)
TOD_I = 1  # 06–12 UTC

# Ensure v27 extended is loaded
if "v27" not in EXT_GAIN:
    EXT_GAIN["v27"] = dict(np.load(C.aggregate_extended("v27", "mr0.00")))

# ── Row 1: Dense MAE ──
dense_mae = []
for vi in SEL_VI:
    cnt = EXT_GAIN["v31"]["tod_mod_cnt"][TOD_I, REF_KI, :, vi]
    s   = EXT_GAIN["v31"]["tod_mod_sum"][TOD_I, REF_KI, :, vi]
    dense_mae.append(np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan))

# ── Row 2: SA gain = MAE(blind) − MAE(dense), positive = Dense better ──
sa_gain = []
for vi in SEL_VI:
    cnt_b = EXT_GAIN["v32-blind"]["tod_mod_cnt"][TOD_I, REF_KI, :, vi]
    s_b   = EXT_GAIN["v32-blind"]["tod_mod_sum"][TOD_I, REF_KI, :, vi]
    mae_b = np.where(cnt_b > 0, s_b / np.maximum(cnt_b, 1), np.nan)
    cnt_d = EXT_GAIN["v31"]["tod_mod_cnt"][TOD_I, REF_KI, :, vi]
    s_d   = EXT_GAIN["v31"]["tod_mod_sum"][TOD_I, REF_KI, :, vi]
    mae_d = np.where(cnt_d > 0, s_d / np.maximum(cnt_d, 1), np.nan)
    sa_gain.append(mae_b - mae_d)  # positive = Dense wins

# ── Row 3: Masking gain = MAE(v27) − MAE(dense), positive = Dense better ──
msk_gain = []
for vi in SEL_VI:
    cnt_v = EXT_GAIN["v27"]["tod_mod_cnt"][TOD_I, REF_KI, :, vi]
    s_v   = EXT_GAIN["v27"]["tod_mod_sum"][TOD_I, REF_KI, :, vi]
    mae_v = np.where(cnt_v > 0, s_v / np.maximum(cnt_v, 1), np.nan)
    cnt_d = EXT_GAIN["v31"]["tod_mod_cnt"][TOD_I, REF_KI, :, vi]
    s_d   = EXT_GAIN["v31"]["tod_mod_sum"][TOD_I, REF_KI, :, vi]
    mae_d = np.where(cnt_d > 0, s_d / np.maximum(cnt_d, 1), np.nan)
    msk_gain.append(mae_v - mae_d)  # positive = Dense wins

ROW_LABELS = ["Dense MAE", "SA gain\n(Blind − Dense)", "Masking cost\n(MAE Tr. − Dense)"]
ROW_DATA   = [dense_mae, sa_gain, msk_gain]
ROW_CMAP   = ["magma", "PRGn", "PRGn"]
NROWS = 3

# Shared colour scale per variable for gain rows
vmax_sa = []
vmax_mk = []
for j in range(N_SV):
    vals_sa = sa_gain[j][~np.isnan(sa_gain[j])]
    vmax_sa.append(np.nanpercentile(np.abs(vals_sa), 98) if vals_sa.size else 1.0)
    vals_mk = msk_gain[j][~np.isnan(msk_gain[j])]
    vmax_mk.append(np.nanpercentile(np.abs(vals_mk), 98) if vals_mk.size else 1.0)

fig, axes = plt.subplots(NROWS, N_SV,
                         figsize=(5.5 * N_SV, 3.5 * NROWS), squeeze=False)
for ri in range(NROWS):
    for ci, (v, vi) in enumerate(zip(SEL_VARS, SEL_VI)):
        ax = axes[ri, ci]
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        vals = ROW_DATA[ri][ci]
        valid = ~np.isnan(vals)

        if ri == 0:  # MAE — sequential
            sc = ax.scatter(stn.easting[valid], stn.northing[valid], c=vals[valid],
                            s=45, cmap="magma", edgecolors="k", linewidths=0.3, zorder=5)
            fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02,
                         label=f"MAE [{C.UNITS[v]}]")
        elif ri == 1:  # SA gain — diverging
            vm = vmax_sa[ci]
            sc = ax.scatter(stn.easting[valid], stn.northing[valid], c=vals[valid],
                            s=45, cmap="PRGn", vmin=-vm, vmax=vm,
                            edgecolors="k", linewidths=0.3, zorder=5)
            fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02,
                         label=f"[{C.UNITS[v]}]")
        else:  # masking gain — diverging
            vm = vmax_mk[ci]
            sc = ax.scatter(stn.easting[valid], stn.northing[valid], c=vals[valid],
                            s=45, cmap="PRGn", vmin=-vm, vmax=vm,
                            edgecolors="k", linewidths=0.3, zorder=5)
            fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02,
                         label=f"[{C.UNITS[v]}]")

        if ri == 0:
            ax.set_title(f"{v} at {REF_LEAD}", fontsize=11)
        if ci == 0:
            ax.text(-0.06, 0.5, ROW_LABELS[ri], transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontsize=9)

# ── Sync spatial axes per variable column ──
for ci in range(N_SV):
    col_axes = [axes[ri, ci] for ri in range(NROWS)]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

plt.tight_layout(h_pad=0.5)
fig.suptitle("Station maps at +3 h for forecasts initialised 06–12 UTC, all stations visible (MR=0)\n"
             "top: Dense MAE — middle: MAE(Spatially Blind) − MAE(Dense) — bottom: MAE(MAE Transformer) − MAE(Dense); green = Dense better",
             fontsize=13, y=1.005)
C.save_fig(fig, "42_morning_mae_sa_msk_maps")
plt.show()
plt.close(fig)


### Error-SD difference on the map, by time of day

Same layout as the MAE gain map above, but showing
σₑ(Blind) − σₑ(Dense) per station per TOD bin.
Positive (green) = Dense has lower error variability.

In [ ]:
# ── Error-SD difference map by time-of-day bin ───────────────────────────────
# σₑ(Blind) − σₑ(Dense) per station, faceted by TOD.
# Reuses EXT_GAIN loaded earlier.

def _station_sd(ext, ti, ki, vi):
    """Per-station error SD for a single (tod, lead, var) slice."""
    sq = ext["tod_mod_sumsq"][ti, ki, :, vi]    # (N,)
    sg = ext["tod_mod_signed"][ti, ki, :, vi]
    cn = ext["tod_mod_cnt"][ti, ki, :, vi]
    with np.errstate(invalid="ignore", divide="ignore"):
        e2 = np.where(cn > 0, sq / cn, np.nan)
        em = np.where(cn > 0, sg / cn, np.nan)
        sd = np.sqrt(np.maximum(e2 - em**2, 0))
    return np.where(cn > 0, sd, np.nan)

sd_diff_by_bin = {}
for ti in range(len(TOD_LABELS)):
    row = []
    for vi in range(NV):
        sd_b = _station_sd(EXT_GAIN["v32-blind"], ti, REF_KI, vi)
        sd_d = _station_sd(EXT_GAIN["v31"], ti, REF_KI, vi)
        row.append(sd_b - sd_d)  # positive = Dense has lower σₑ
    sd_diff_by_bin[ti] = row

vmax_sd = []
for vi in range(NV):
    allv = np.concatenate([sd_diff_by_bin[ti][vi][
        ~np.isnan(sd_diff_by_bin[ti][vi])] for ti in range(len(TOD_LABELS))])
    vmax_sd.append(np.nanpercentile(np.abs(allv), 98) if allv.size else 1.0)

fig, axes = plt.subplots(len(TOD_LABELS), NV,
                         figsize=(6.0 * NV, 3.5 * len(TOD_LABELS)), squeeze=False)
for ti, tod_label in enumerate(TOD_LABELS):
    for vi, v in enumerate(VARS):
        ax = axes[ti, vi]
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        diff = sd_diff_by_bin[ti][vi]
        valid = ~np.isnan(diff)
        vmax = vmax_sd[vi]
        sc = ax.scatter(stn.easting[valid], stn.northing[valid], c=diff[valid],
                        s=45, cmap="PRGn", vmin=-vmax, vmax=vmax,
                        edgecolors="k", linewidths=0.3, zorder=5)
        overlay_exclusion_markers(ax, v, legend=(ti == 0 and vi == 0))
        fig.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label=f"[{C.UNITS[v]}]")
        if ti == 0:
            ax.set_title(f"{v} at {REF_LEAD}", fontsize=11)
        if vi == 0:
            ax.text(-0.06, 0.5, tod_label, transform=ax.transAxes, rotation=90,
                   va="center", ha="center", fontsize=10)

fig.suptitle("Error-SD difference by station and time of day\n"
             "σₑ(Blind) − σₑ(Dense): green = spatial attention reduces variability (MR=0)",
             y=1.005)
plt.tight_layout()
# ── Sync y-axis per variable column across rows ──
for vi in range(NV):
    col_axes = [axes[ri, vi] for ri in range(len(TOD_LABELS))]
    lo = min(ax.get_ylim()[0] for ax in col_axes)
    hi = max(ax.get_ylim()[1] for ax in col_axes)
    for ax in col_axes:
        ax.set_ylim(lo, hi)

# ── Sync wind component color scales ──
wu_i, wv_i = VARS.index("wind_u"), VARS.index("wind_v")
for ri in range(axes.shape[0]):
    sc_u = axes[ri, wu_i].collections[-1]
    sc_v = axes[ri, wv_i].collections[-1]
    vmin = min(sc_u.get_clim()[0], sc_v.get_clim()[0])
    vmax = max(sc_u.get_clim()[1], sc_v.get_clim()[1])
    sc_u.set_clim(vmin, vmax)
    sc_v.set_clim(vmin, vmax)
C.save_fig(fig, "42_spatial_attn_sd_diff_map_tod"); plt.show()
plt.close(fig)


## Stations excluded from evaluation, by variable

Marks every station×variable pair dropped by `DROP_SV` (see common.py) with
an X on the map: **black** = missing in both train and test (>50%),
**white** = missing in train only (sensor added after 2021, present in
test), **grey** = missing in test only (no such pair exists in the current
data — kept for completeness), **orange** = present in both splits but
excluded for a data-quality reason (BIZ/pressure, systematic sensor drift),
not for missingness.